Time Series: PatchTST, N-BEATS, Chronos, and Probabilistic Forecasting

Deep learning architectures and probabilistic paradigms for time-series forecasting:
1. Reversible Instance Normalization (RevIN): Mitigating distribution drift via symmetric normalization/denormalization.
2. PatchTST: Channel-independent patching and linear embedding for long lookback horizons.
3. N-BEATS: Deep doubly residual stacking of backcast and forecast polynomial/seasonality blocks.
4. Probabilistic Forecasting: Quantile regression and pinball loss across multiple confidence intervals.

## 2. Mathematical Formulations & Theoretical Foundations

- **RevIN Normalization & Denormalization**:
  $$\hat{X} = \frac{X - \mu}{\sqrt{\sigma^2 + \epsilon}} \cdot \gamma + \beta, \quad Y = \left(\frac{\hat{Y} - \beta}{\gamma}\right) \sqrt{\sigma^2 + \epsilon} + \mu$$
- **PatchTST Patching**:
  $$N_{\text{patches}} = \left\lfloor \frac{L - P}{S} \right\rfloor + 1, \quad X_{\text{unfolded}} \in \mathbb{R}^{(B \cdot M) \times N \times P}$$
- **N-BEATS Doubly Residual Block**:
  $$x_\ell = x_{\ell-1} - g^b(\theta_\ell^b), \quad \hat{y} = \sum_{\ell=1}^K g^f(\theta_\ell^f)$$
- **Pinball (Quantile) Loss**:
  $$\mathcal{L}_q(y, \hat{y}_q) = \max\left(q(y - \hat{y}_q), (1 - q)(\hat{y}_q - y)\right)$$

In [ ]:
import math
from pathlib import Path
import torch
import torch.nn.functional as F
from safetensors.torch import load_file

torch.manual_seed(42)
REPO_ROOT = next(p for p in Path.cwd().parents if (p / "Cargo.toml").exists()) if not (Path.cwd() / "Cargo.toml").exists() else Path.cwd()
GOLDEN_PATH = REPO_ROOT / "weights" / "07_domain_tasks" / "09_time_series_patchtst_nbeats_chronos_and_probabilistic.safetensors"
golden = load_file(str(GOLDEN_PATH))
print(f"Loaded {len(golden)} tensors from golden checkpoint.")

In [ ]:
# 1. Reversible Instance Normalization (RevIN)
def revin_normalize(x: torch.Tensor, gamma: torch.Tensor, beta: torch.Tensor, eps: float = 1e-5):
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, keepdim=True, unbiased=False)
    stdev = torch.sqrt(var + eps)
    x_norm = ((x - mean) / stdev) * gamma + beta
    return x_norm, mean, stdev

def revin_denormalize(y: torch.Tensor, mean: torch.Tensor, stdev: torch.Tensor, gamma: torch.Tensor, beta: torch.Tensor):
    return ((y - beta) / gamma) * stdev + mean

norm_x, mean, stdev = revin_normalize(golden["revin_x"], golden["revin_gamma"], golden["revin_beta"])
denorm_y = revin_denormalize(golden["norm_y"], mean, stdev, golden["revin_gamma"], golden["revin_beta"])

diff_norm = torch.max(torch.abs(norm_x - golden["revin_norm_x"])).item()
diff_denorm = torch.max(torch.abs(denorm_y - golden["denorm_y_ref"])).item()
print(f"RevIN Normalize max abs error: {diff_norm:.2e}")
print(f"RevIN Denormalize max abs error: {diff_denorm:.2e}")
assert diff_norm < 1e-5
assert diff_denorm < 1e-5

In [ ]:
# 2. PatchTST Patching & Channel-Independence Unfolding
def patchtst_patching(x: torch.Tensor, patch_len: int, stride: int) -> torch.Tensor:
    b, m, l = x.shape
    patches = x.unfold(dimension=-1, size=patch_len, step=stride)
    n_patches = patches.shape[2]
    return patches.contiguous().view(b * m, n_patches, patch_len)

patches_calc = patchtst_patching(golden["revin_x"], patch_len=8, stride=4)
diff_patches = torch.max(torch.abs(patches_calc - golden["patches_ref"])).item()
print(f"PatchTST Patching max abs error: {diff_patches:.2e}")
assert diff_patches < 1e-5

In [ ]:
# 3. N-BEATS Doubly Residual Block
def nbeats_block(
    x: torch.Tensor,
    fc_weights: list[torch.Tensor],
    fc_biases: list[torch.Tensor],
    theta_b_weight: torch.Tensor,
    theta_f_weight: torch.Tensor,
    basis_b: torch.Tensor,
    basis_f: torch.Tensor,
):
    h = x
    for w, b in zip(fc_weights, fc_biases):
        h = F.relu(h @ w + b)
    theta_b = h @ theta_b_weight
    theta_f = h @ theta_f_weight
    backcast = theta_b @ basis_b
    forecast = theta_f @ basis_f
    return backcast, forecast

backcast_calc, forecast_calc = nbeats_block(
    golden["nbeats_x"],
    [golden["w_fc1"], golden["w_fc2"]],
    [golden["b_fc1"], golden["b_fc2"]],
    golden["w_theta_b"],
    golden["w_theta_f"],
    golden["basis_b"],
    golden["basis_f"],
)
diff_backcast = torch.max(torch.abs(backcast_calc - golden["backcast_ref"])).item()
diff_forecast = torch.max(torch.abs(forecast_calc - golden["forecast_ref"])).item()
print(f"N-BEATS Backcast max abs error: {diff_backcast:.2e}")
print(f"N-BEATS Forecast max abs error: {diff_forecast:.2e}")
assert diff_backcast < 1e-5
assert diff_forecast < 1e-5

In [ ]:
# 4. Probabilistic Forecasting: Pinball Quantile Loss & Summary
def pinball_quantile_loss(y_true: torch.Tensor, y_pred: torch.Tensor, quantiles: torch.Tensor) -> torch.Tensor:
    err = y_true.unsqueeze(1) - y_pred
    q = quantiles.view(1, -1, 1)
    loss = torch.maximum(q * err, (q - 1.0) * err)
    return loss.mean()

pinball_calc = pinball_quantile_loss(golden["y_true"], golden["y_pred_quantiles"], golden["quantiles"])
diff_pinball = torch.abs(pinball_calc - golden["pinball_loss_ref"]).item()

print("=" * 65)
print(f"{'Operator':<32} | {'Max Abs Error':<15} | Status")
print("-" * 65)
print(f"{'RevIN Normalize':<32} | {diff_norm:<15.2e} | PASSED")
print(f"{'RevIN Denormalize':<32} | {diff_denorm:<15.2e} | PASSED")
print(f"{'PatchTST Patching':<32} | {diff_patches:<15.2e} | PASSED")
print(f"{'N-BEATS Backcast':<32} | {diff_backcast:<15.2e} | PASSED")
print(f"{'N-BEATS Forecast':<32} | {diff_forecast:<15.2e} | PASSED")
print(f"{'Pinball Quantile Loss':<32} | {diff_pinball:<15.2e} | PASSED")
print("=" * 65)
assert diff_pinball < 1e-5

## 8. Summary, Invariants & Diagnostic Checklist

- [x] Reversible Instance Normalization preserves seasonal dynamics while decoupling non-stationary drift.
- [x] PatchTST subseries tokenization avoids quadratic overhead and enhances signal-to-noise ratio.
- [x] N-BEATS hierarchical backcast/forecast residuals enable interpretable polynomial and harmonic decomposition.
- [x] Pinball quantile losses produce calibrated uncertainty intervals for multi-horizon risk quantification.